# Level 0 multi-seed uncertainty bands


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path(os.getenv('NANOGPT_LEVEL0_RESULTS_ROOT','/tmp/nanogpt-level0/results'))
rows=[]
for path in ROOT.glob('*_seed_*/metrics.csv'):
    optimizer,seed=path.parent.name.split('_seed_')
    d=pd.read_csv(path); d['optimizer']=optimizer; d['seed']=int(seed)
    for split in ['train','val','test']: d[f'{split}_error']=1-d[f'{split}_accuracy']
    rows.append(d)
all_df=pd.concat(rows,ignore_index=True)
all_df.groupby('optimizer').seed.nunique()


In [ ]:
def band_plot(metric):
    plt.figure(figsize=(10,6))
    for optimizer,d in all_df.groupby('optimizer'):
        a=d.groupby('step')[metric].agg(['mean','std']).reset_index(); s=a['std'].fillna(0)
        line,=plt.plot(a.step,a['mean'],label=optimizer)
        plt.fill_between(a.step,a['mean']-s,a['mean']+s,alpha=.2,color=line.get_color())
    plt.xlabel('step'); plt.ylabel(metric); plt.title(f'{metric}: mean ± 1 standard deviation'); plt.legend(); plt.grid(alpha=.25); plt.show()
for metric in ['test_loss','test_accuracy','test_error','test_perplexity','test_generalization_gap']: band_plot(metric)


In [ ]:
ww=[]
for run in ROOT.glob('*_seed_*'):
    optimizer,seed=run.name.split('_seed_')
    for f in run.glob('weightwatcher_step_*.csv'):
        d=pd.read_csv(f); d['optimizer']=optimizer; d['seed']=int(seed); ww.append(d)
if ww:
    ww=pd.concat(ww,ignore_index=True); layer_col='layer_id' if 'layer_id' in ww else 'layer'
    for layer,ld in ww.groupby(layer_col):
        plt.figure(figsize=(10,5))
        for optimizer,d in ld.groupby('optimizer'):
            a=d.groupby('step').alpha.agg(['mean','std']).reset_index(); s=a['std'].fillna(0)
            line,=plt.plot(a.step,a['mean'],label=optimizer); plt.fill_between(a.step,a['mean']-s,a['mean']+s,alpha=.2,color=line.get_color())
        plt.title(f'Layer {layer} alpha'); plt.legend(); plt.show()
